In [2]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
import pandas as pd

In [3]:
from pathlib import Path

In [4]:
DATA_ROOT=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
path=DATA_ROOT/"simulated/shendure_pow_analysis"
name="sim_with_orthos_20251206"

In [5]:
spread_hypothesis_20251119=scm.HypothesisSet.from_tsv(path/"spread_hypothesis_20251119.tsv")

In [7]:
from dask.distributed import get_client#Semaphore, as_completed,

In [8]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=4,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=2:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=1)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

client.dashboard_link

'http://127.0.0.1:8787/status'

2025-12-10 10:51:15,878 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='a1132u05n01.mghpcc.ycrc.yale.edu:25595', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/tornado/web.py", line 3375, in wrapper
    return method(self, *args, **kwargs)
  File "/home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expi

# try it erin's way

In [15]:
ortho_root=path/name/"orthos_with_precomputed_wald"
output_root=path/name/"results"
output_root.mkdir(exist_ok=True,parents=True)
input_ortho_names=[path.name for path in ortho_root.iterdir()]

name_oi = input_ortho_names[0]

ortho_oi=scm.ortho.load(client,ortho_root,name_oi)
scmdata_oi=ortho_oi.training_data

In [17]:
scmdata_oi.data

,rep_id,cell_bc,cre_id_original,r,cell_type,mu,zi,sigmasquare,mpra_bc,p,cre_id,theta,umis_mpra_bc
0,2B1,AAAAAAAAAAAAAAAAAAAA,active_0,0.354348,Cardiomyocytes,163.787951,0.019746,75870.518561,AAAAAAAAAAAAACGGCCGA,0.002159,active_0,0.354348,0
1,2B1,AAAAAAAAAAAAAAAAAAAA,active_1,0.354348,Cardiomyocytes,74.728005,0.019746,15834.044101,AAAAAAAAAAAAACGTACCT,0.004719,active_1,0.354348,102
2,2B1,AAAAAAAAAAAAAAAAAAAA,active_11,0.354348,Cardiomyocytes,105.882041,0.019746,31744.33231,AAAAAAAAAAAAACTTTCGA,0.003335,active_11,0.354348,99
3,2B1,AAAAAAAAAAAAAAAAAAAA,active_14,0.354348,Cardiomyocytes,262.628109,0.019746,194912.000459,AAAAAAAAAAAAAGACAAGT,0.001347,active_14,0.354348,2377
4,2B1,AAAAAAAAAAAAAAAAAAAA,active_14,0.354348,Cardiomyocytes,262.628109,0.019746,194912.000459,AAAAAAAAAAAAAGACAGGG,0.001347,active_14,0.354348,187
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4673182,B2,AAAAAAAAAAAAGGGCCTTA,inactive_23,0.354348,reference,0.01935,0.019881,0.020407,AAAAAAAAAAAAAATAGTGT,0.948219,inactive_23,0.354348,0
4673183,B2,AAAAAAAAAAAAGGGCCTTA,inactive_35,0.354348,reference,0.01935,0.019881,0.020407,AAAAAAAAAAAAACATGCTT,0.948219,inactive_35,0.354348,0
4673184,B2,AAAAAAAAAAAAGGGCCTTA,inactive_39,0.354348,reference,0.01935,0.019881,0.020407,AAAAAAAAAAAAACCCACCG,0.948219,inactive_39,0.354348,0
4673185,B2,AAAAAAAAAAAAGGGCCTTA,inactive_48,0.354348,reference,0.01935,0.019881,0.020407,AAAAAAAAAAAAACGCACGA,0.948219,inactive_48,0.354348,0


In [22]:
runner = scm.HypothesisTester("wald")
wald_by_ct  = runner.run(ct_spread_hypothesis_20251119, ortho_oi)
wald_ct_df = wald_by_ct.to_dataframe()
wald_ct_df

,Unnamed: 0,comparison_CRE,reference_CRE,comparison_cell_type,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,wald_debug,test_type,bh_p
0,0,inactive_0,reference,Cardiomyocytes,Cardiomyocytes,<NA>,NaN,NaN,NaN,False,no contrast term for inactive_0,wald,1.000000e+00
1,1,inactive_0,reference,EpiblastPrimitiveStreak,EpiblastPrimitiveStreak,<NA>,1.185062,2.359930e-01,1.449675,False,ok,wald,3.511801e-01
2,2,inactive_0,reference,ExEndodermParietal,ExEndodermParietal,<NA>,0.072885,9.418980e-01,1.021343,False,ok,wald,9.765995e-01
3,3,inactive_0,reference,ExEndodermVisceral,ExEndodermVisceral,<NA>,1.175322,2.398659e-01,1.496037,False,ok,wald,3.549736e-01
4,4,inactive_0,reference,Haematoendothelial,Haematoendothelial,<NA>,0.092704,9.261391e-01,1.045617,False,ok,wald,9.678787e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,active_49,reference,Mesoderm,Mesoderm,<NA>,82.037645,2.225074e-308,5913.548619,False,ok,wald,5.285211e-308
996,996,active_49,reference,NeuroectodermBrain,NeuroectodermBrain,<NA>,86.209752,2.225074e-308,5295.186555,False,ok,wald,5.285211e-308
997,997,active_49,reference,NeuroectodermRostral,NeuroectodermRostral,<NA>,36.480915,2.226291e-291,7070.669169,False,ok,wald,5.141550e-291
998,998,active_49,reference,SurfaceEctoderm,SurfaceEctoderm,<NA>,69.552818,2.225074e-308,5872.438510,False,ok,wald,5.285211e-308


# mackenzie's og code

In [8]:
ortho_root=path/name/"orthos_with_precomputed_wald"
output_root=path/name/"results"
output_root.mkdir(exist_ok=True,parents=True)
input_ortho_names=[path.name for path in ortho_root.iterdir()]

#Semaphore(max_leases=10, name="test")

def compute_one_wald(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    #sem = Semaphore(name="test")

    client=get_client()
    ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
    scmpradat_oi=scm.scMPRA_data.from_parquet(path/"sim_with_orthos_20251206"/"scMPRA"/f"{name}.scmpra")
    ortho_oi.training_data=scmpradat_oi
    runner = scm.HypothesisTester(test_type)
    output_short=Path(output_root)/hypothesis_set_name/test_type
    output_short.mkdir(exist_ok=True,parents=True)
    #FOR WALD
    runner.run(hypothesis_set, ortho_oi, client).to_tsv(output_short/name)
    #FOR MWU
    #runner.run(hypothesis_set, scmpradat_oi, client).to_tsv(output_short/name)

'http://127.0.0.1:8787/status'

In [ ]:
a = client.submit(compute_one_test,
                       input_root=ortho_root,
                       name='0',
                       output_root=output_root,
                       hypothesis_set=ct_spread_hypothesis_20251119,
                       hypothesis_set_name="spread_hypothesis_20251119",
                       test_type="wald", use_client=True)

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']


In [13]:
a.result

<bound method Future.result of <Future: pending, key: compute_one_test-9fa6878af78828e950dc75701ba600df>>

In [ ]:
futures_wald = [client.submit(compute_one_test,
                       input_root=ortho_root,
                       name=name_oi,
                       output_root=output_root,
                       hypothesis_set=ct_spread_hypothesis_20251119,
                       hypothesis_set_name="spread_hypothesis_20251119",
                       test_type="wald") for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [31]:
futures_wald

[<Future: finished, type: NoneType, key: compute_one_test-5e3e763195cdd6a771cacfe64fd674bb>,
 <Future: pending, key: compute_one_test-69d9901342520e0975affb32efe0a680>,
 <Future: pending, key: compute_one_test-02da806b789aac25550480312ec1bbd1>,
 <Future: pending, key: compute_one_test-2ec3cdc8ee6e655e89e553e815ba57d1>,
 <Future: pending, key: compute_one_test-e2d7e557a6bce814c7b2c192d502a760>]

In [ ]:
futures_mwu = [client.submit(compute_one_test,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=spread_hypothesis_20251119,
                        hypothesis_set_name="spread_hypothesis_20251119",
                        test_type="mwu", use_client=True) for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [ ]:
results = [compute_one_test(
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=spread_hypothesis_20251119,
                        hypothesis_set_name="spread_hypothesis_20251119",
                        test_type="mwu") for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [ ]:
futures_mwu[0].result()

In [ ]:
client.dashboard_link

In [10]:
client.close()
cluster.close()

2025-12-08 17:14:27,062 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-cfabaed90a31ded0f3060faafa8eac4b')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-08 17:14:27,062 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-dc6f04e2d4c1e5e26510aff7ee081df3')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-08 17:14:27,063 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-db8fc46e2b9ba7561fb2352328990c52')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machin